In [6]:
!pip install imbalanced-learn xgboost joblib -q

In [7]:
from google.colab import files

uploaded = files.upload()

Saving diabetes.csv to diabetes.csv


In [8]:
import pandas as pd

df = pd.read_csv(list(uploaded.keys())[0])
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
!pip install imbalanced-learn xgboost joblib -q

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_csv("diabetes.csv")

print("="*60)
print("DATASET SHAPE")
print("="*60)

print(df.shape)

# ==========================================================
# DATASET INFORMATION
# ==========================================================

print("\nDATASET INFO")
print("="*60)

print(df.info())

print("\nMISSING VALUES")
print("="*60)

print(df.isnull().sum())

print("\nSTATISTICAL SUMMARY")
print("="*60)

print(df.describe())

# ==========================================================
# CORRELATION HEATMAP
# ==========================================================

plt.figure(figsize=(10,8))

sns.heatmap(
    df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Matrix")

plt.show()

# ==========================================================
# CLASS DISTRIBUTION BEFORE SMOTE
# ==========================================================

print("\nCLASS DISTRIBUTION")
print("="*60)

print(df['Outcome'].value_counts())

plt.figure(figsize=(5,4))

sns.countplot(
    x='Outcome',
    data=df
)

plt.title("Before SMOTE")

plt.show()

# ==========================================================
# FEATURES & TARGET
# ==========================================================

X = df.drop("Outcome", axis=1)

y = df["Outcome"]

# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================================================
# SMOTE
# ==========================================================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nAFTER SMOTE")
print("="*60)

print(pd.Series(y_train_smote).value_counts())

plt.figure(figsize=(5,4))

sns.countplot(
    x=y_train_smote
)

plt.title("After SMOTE")

plt.show()

# ==========================================================
# FEATURE SCALING
# ==========================================================

scaler = StandardScaler()

X_train_smote = scaler.fit_transform(
    X_train_smote
)

X_test = scaler.transform(
    X_test
)

# ==========================================================
# MODELS
# ==========================================================

models = {

    "Logistic Regression":
    LogisticRegression(
        max_iter=2000
    ),

    "SVM":
    SVC(
        kernel='rbf',
        probability=True
    ),

    "Random Forest":
    RandomForestClassifier(
        n_estimators=500,
        max_depth=15,
        random_state=42
    ),

    "XGBoost":
    XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        random_state=42,
        eval_metric='logloss'
    )
}

# ==========================================================
# TRAIN & EVALUATE
# ==========================================================

results = []

best_model_name = None
best_model_object = None
best_accuracy = 0

for name, model in models.items():

    print("\n")
    print("="*70)
    print(name)
    print("="*70)

    model.fit(
        X_train_smote,
        y_train_smote
    )

    y_pred = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    # ROC AUC

    y_prob = model.predict_proba(
        X_test
    )[:,1]

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    print(
        f"Accuracy : {accuracy*100:.2f}%"
    )

    print(
        f"ROC-AUC  : {auc:.4f}"
    )

    print("\nClassification Report")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )

    # Confusion Matrix

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    plt.figure(figsize=(5,4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(
        f"Confusion Matrix - {name}"
    )

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.show()

    results.append(
        [
            name,
            accuracy,
            auc
        ]
    )

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        best_model_name = name

        best_model_object = model

# ==========================================================
# MODEL COMPARISON TABLE
# ==========================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "ROC_AUC"
    ]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n")
print("="*70)
print("MODEL COMPARISON")
print("="*70)

print(results_df)

# ==========================================================
# MODEL COMPARISON GRAPH
# ==========================================================

plt.figure(figsize=(8,5))

sns.barplot(
    data=results_df,
    x="Accuracy",
    y="Model"
)

plt.title(
    "Model Accuracy Comparison"
)

plt.show()

# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

print("\n")
print("="*70)
print("FEATURE IMPORTANCE (RANDOM FOREST)")
print("="*70)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    random_state=42
)

rf.fit(
    X_train_smote,
    y_train_smote
)

importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance":
    rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

plt.figure(figsize=(8,5))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title(
    "Feature Importance"
)

plt.show()

# ==========================================================
# SAVE BEST MODEL
# ==========================================================

joblib.dump(
    best_model_object,
    "best_diabetes_model.pkl"
)

print("\n")
print("="*70)
print("BEST MODEL")
print("="*70)

print("Model Name :", best_model_name)

print(
    f"Accuracy   : {best_accuracy*100:.2f}%"
)

print(
    "\nModel saved as best_diabetes_model.pkl"
)